In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import ExtraTreeClassifier, DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import confusion_matrix, plot_confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from ast import literal_eval
from tqdm import tqdm

In [ ]:
tqdm.pandas()

In [ ]:
songs = pd.read_csv('./data/five_g_500_modeling.csv')

In [ ]:
features = songs.columns[22:].drop([])

In [ ]:
X = songs[features]
y = songs['genre']

In [ ]:
def running_models(X, y, scaled=False, c=10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=182)
    
    # MODELING WITH SCALED FEATURES #
    
    # Scale the features
    sc = StandardScaler()
    X_train_sc = sc.fit_transform(X_train)
    X_test_sc = sc.transform(X_test)
    
    # Creating a df to display results
    model_df = pd.DataFrame(columns=['train_score', 'test_score'])
    
    model_sc = SVC(C=c, kernel='rbf', degree=3)
    model_sc.fit(X_train_sc, y_train)
    train_score = model_sc.score(X_train_sc, y_train)
    test_score = model_sc.score(X_test_sc, y_test)
    
    row = {'model': type(model_sc).__name__,
           'train_score' : train_score,
          'test_score' : test_score,
           'C': c,
          'scaled': True}
    model_df = model_df.append(row, ignore_index=True)
    
    # MODELING WITH UNSCALED FEATURES #
    
    model = SVC(C=c, kernel='rbf', degree=3)
    model.fit(X_train, y_train)
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    
    row = {'model': type(model).__name__,
       'train_score' : train_score,
      'test_score' : test_score,
       'C': c,
      'scaled': False}
    model_df = model_df.append(row, ignore_index=True)
    return model_df

There's a high **positive correlation between loudness and energy** and a high **negative correlation between energy and acousticness**

In [ ]:
running_models(X, y)

In [ ]:
# Create train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=182)

# Scale the features
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [ ]:
model = SVC(C=.35)
model.fit(X_train_sc, y_train)

In [ ]:
categories = list(set(y_preds))
numbers = list(range(5))

In [ ]:
y_preds = model.predict(X_test_sc)

In [ ]:
results = X_test.copy()

In [ ]:
results.loc[:, 'genre'] = y_test.copy()

In [ ]:
results.loc[:, 'preds'] = y_preds.copy()

In [ ]:
plot_confusion_matrix(model, X_test_sc, y_test, xticks_rotation='vertical', normalize='true');

We can see here that our model performs **best on classical music**, and performs **worst on r&b, only predicting correctly 74% of the time.** I want to dig more into why that is

In [ ]:
results_wrong = results[results['genre'] != results['preds']]

In [ ]:
songs_wrong = songs.loc[results_wrong.index, ['track_name']]

In [ ]:
results_wrong['track_name'] = songs_wrong

In [ ]:
results_wrong[(results_wrong['preds'] == 'r&b') & (results_wrong['genre'] == 'rap')].head()

In [ ]:
group_preds = results_wrong.groupby('preds').mean()

In [ ]:
# Getting columns where rap and r&b are very close to each other
group_preds.columns[abs(group_preds.loc['rap']) - abs(group_preds.loc['r&b']) < 0.001]

In [ ]:
results_wrong['genre'].value_counts(normalize=True)

Rap and R&B seem to be predicted wrong most often with these new features I've pulled

In [ ]:
results_wrong[results_wrong['genre'] == 'rap']['preds'].value_counts(normalize=True)

It seems that when Rap is predicted wrong, it's predicted as r&b.

In [ ]:
results_wrong[results_wrong['genre'] == 'r&b']['preds'].value_counts(normalize=True)

When R&B is predicted wrong, it's predicted to be rap 84% of the time

In [ ]:
genre_mean = songs.groupby('genre').mean()

Interesting that most of the genres hover slightly above 120 as a tempo. There's some interesting reasoning behind this with commercial music and the need for 30-seconds of music to fill an advertising slot. 128 beats per minute would allow for a 16-bar phrase in 4/4 time.

In [ ]:
X = songs[features]
# Thanks joris on SO: https://stackoverflow.com/questions/32011359/convert-categorical-data-in-pandas-dataframe/32011969
y_cat = songs['genre'].astype('category').cat.codes

In [ ]:
y = to_categorical(y_cat)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=182)

In [ ]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [ ]:
fnn_model = Sequential()

fnn_model.add(Dense(16, activation='relu', input_shape=(X_train_sc.shape[1],)))
fnn_model.add(Dense(64, activation='relu'))
fnn_model.add(Dropout(0.2))
fnn_model.add(Dense(32, activation='relu'))
fnn_model.add(Dropout(0.2))
fnn_model.add(Dense(16, activation='relu'))
fnn_model.add(Dense(5, activation='softmax'))

fnn_model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['acc'])

hist = fnn_model.fit(X_train_sc, y_train,
                epochs=100,
                batch_size=128,
                validation_data=(X_test_sc, y_test),
                    use_multiprocessing=True,
                    verbose=1)

In [ ]:
plt.plot(hist.history['acc'], label="Training Acc")
plt.plot(hist.history['val_acc'], label="Testing Acc")
plt.legend();

With my extracted features (energy, rms, and mfcc), I'm able to improve on the accuracy of the spotify model by **5-7% on training data**
* SVC model $\approx 84\%$ accuracy
* FNN model $\approx 82\%$ accuracy

## What's next?

I still want to find some way for the model to be able to tell the difference between rap and r&b